# AFC Chat example: use Chat.send_message with automatic function calling (AFC)
#
# This notebook demonstrates sending messages via the Chat API, handling a `function_call`,
# executing a local function, and returning its result back to the chat.
# Note: set GOOGLE_API_KEY in a .env file or environment variable before running cells.

In [ ]:
# If you need to install dependencies run in a terminal:
# pip install -r requirements.txt

import os
import json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

In [ ]:
chat = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    api_key=os.environ.get("GOOGLE_API_KEY"),
)
print("Chat client created:", type(chat))

In [ ]:
function_defs = [
    {
        "name": "get_weather",
        "description": "Return current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "units": {"type": "string", "enum": ["metric", "imperial"]}
            },
            "required": ["city"]
        },
    }
]

# simple stub implementation

def get_weather(city, units="metric"):
    return {"city": city, "temperature": 20, "units": units}

In [ ]:
user_message = {"role": "user", "content": "What's the weather in Paris in metric?"}

# 1) Send message via Chat.send_message
resp = chat.send_message(messages=[user_message], function_definitions=function_defs)

# 2) Inspect response for a function_call
# Adjust attribute access if your SDK returns an object instead of a dict
func_call = None
if isinstance(resp, dict):
    func_call = resp.get("function_call")
else:
    func_call = getattr(resp, "function_call", None)

if func_call:
    name = func_call["name"]
    args = json.loads(func_call.get("arguments", "{}"))
    print("Model requested function:", name, args)

    # 3) Execute the function locally
    result_obj = get_weather(**args)
    result_text = json.dumps(result_obj)

    # 4) Send the function result back into the chat as a `function` role message
    followup_messages = [user_message, {"role": "function", "name": name, "content": result_text}]
    final = chat.send_message(messages=followup_messages)
    print("Final model response:", final)
else:
    print("Model response (no function call):", resp)

## Notes

- Replace `get_weather` with a real API call to fetch live data.
- If `resp` is an SDK object, inspect its attributes (e.g., `resp.function_call`) rather than dict keys.
- Ensure `GOOGLE_API_KEY` is set in a `.env` file or exported in the shell before running cells.